# AI-Based Smart Logistics Platform - Model Training Notebook

This Google Colab notebook mounts Google Drive, searches recursively for `flood_risk_dataset_india.csv` inside `/content/drive/MyDrive/`, executes data preprocessing, evaluates Baseline Logistic Regression, Random Forest, and XGBoost models, and exports model artifacts (`risk_model.pkl`, `preprocessor.pkl`, `model_metadata.json`).

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Local execution environment detected.')

In [ ]:
import os
import pandas as pd
import numpy as np

# Search recursively for dataset in /content/drive/MyDrive/ or current folder
search_paths = ['/content/drive/MyDrive', '.']
dataset_path = None
for base in search_paths:
    if os.path.exists(base):
        for root, _, files in os.walk(base):
            for f in files:
                if 'flood_risk_dataset' in f.lower() and f.endswith('.csv'):
                    dataset_path = os.path.join(root, f)
                    break
            if dataset_path:
                break

print('Dataset path found:', dataset_path)
df = pd.read_csv(dataset_path or 'flood_risk_dataset_india.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import joblib, json, datetime

X = df.drop(columns=['Flood Occurred'])
y = df['Flood Occurred']

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

X_trans = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_trans, y, test_size=0.2, random_state=42, stratify=y)

# Models
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42).fit(X_train, y_train)
xgb = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42).fit(X_train, y_train)

print('LR F1:', f1_score(y_test, lr.predict(X_test)))
print('RF F1:', f1_score(y_test, rf.predict(X_test)))
print('XGB F1:', f1_score(y_test, xgb.predict(X_test)))